# 05 - Modelado: Clustering de Clientes con K-Means

## Objetivo
Agrupar clientes según su comportamiento RFM usando K-Means.
El K óptimo es determinado por el notebook 05 (Silhouette Score + Método del Codo).

Fuente: `bigdata.default.gold_customer_features`

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings("ignore")

# Leer tabla Gold (capa Medallón)
df_gold = spark.table("bigdata.default.gold_customer_features")
print("Registros en Gold:", df_gold.count())
df_gold.show(5)

## Parte 2 — Conversión a Pandas y Escalado (CRÍTICO)

Antes de K-Means aplicamos StandardScaler porque las variables están en escalas muy distintas:
- `recency` → días (números grandes)
- `frequency` → conteo (números pequeños)
- `monetary` → valores monetarios (números muy grandes)

Sin escalar, `monetary` dominaría el modelo y distorsionaría los clústeres.
StandardScaler transforma cada variable a media=0 y desviación estándar=1.

In [0]:
# Convertir a Pandas
pdf = df_gold.select("id_cliente", "recency", "frequency", "monetary").toPandas()

print(f"Clientes únicos: {len(pdf):,}")
print("\nEstadísticas descriptivas (escala original):")
print(pdf[["recency", "frequency", "monetary"]].describe().round(2))

In [0]:
# Escalar con StandardScaler (media=0, std=1)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(pdf[["recency", "frequency", "monetary"]])

print("Escalado completado.")
print(f"Media post-escalado  : {rfm_scaled.mean(axis=0).round(4)}")
print(f"StdDev post-escalado : {rfm_scaled.std(axis=0).round(4)}")

## Parte 3 — Carga del K Óptimo y Entrenamiento

El K óptimo fue determinado en el notebook 05 evaluando:
- Método del Codo (inercia)
- Silhouette Score

Si el archivo no existe, se usa K=4 como valor por defecto.

In [0]:
# Leer K óptimo generado por notebook 05
try:
    with open("/tmp/kmeans_config.json", "r") as f:
        k_config = json.load(f)
    optimal_k  = k_config["optimal_k"]
    silhouette = k_config["silhouette_score"]
    print(f"K óptimo cargado desde notebook 05: K={optimal_k}  (Silhouette={silhouette:.4f})")
except FileNotFoundError:
    optimal_k = 4
    print(f"Archivo no encontrado — usando K default: {optimal_k}")

# Entrenar modelo final
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
pdf["cluster"] = km_final.fit_predict(rfm_scaled)

print(f"\n✅ Modelo K-Means entrenado con K={optimal_k}")
print(f"Inercia final: {km_final.inertia_:,.1f}")

## Parte 4 — Análisis de Centroides

Los centroides muestran el "cliente típico" de cada clúster en escala normalizada.
- Valor positivo → por encima del promedio global
- Valor negativo → por debajo del promedio global

In [0]:
# Centroides en escala normalizada
centroids = km_final.cluster_centers_
print("Centroides por clúster (escala normalizada):")
print(f"{'Cluster':<10} {'Recency':>12} {'Frequency':>12} {'Monetary':>12}")
print("-" * 50)
for i, c in enumerate(centroids):
    print(f"Cluster {i:<3}  {c[0]:>12.3f} {c[1]:>12.3f} {c[2]:>12.3f}")

# Centroides en escala original (más interpretable)
centroids_original = scaler.inverse_transform(centroids)
print("\nCentroides por clúster (escala original):")
print(f"{'Cluster':<10} {'Recency (días)':>16} {'Frequency':>12} {'Monetary ($)':>14}")
print("-" * 56)
for i, c in enumerate(centroids_original):
    print(f"Cluster {i:<3}  {c[0]:>16.1f} {c[1]:>12.1f} {c[2]:>14.2f}")

In [0]:
# Distribución de clientes por clúster
print("Distribución de clientes por clúster:")
print(pdf["cluster"].value_counts().sort_index().to_string())
print(f"\nTotal clientes segmentados: {len(pdf):,}")

In [0]:
# Vista de muestra de clientes con su clúster asignado
print("Muestra de clientes con clúster asignado:")
print(pdf[["id_cliente", "recency", "frequency", "monetary", "cluster"]].head(10).to_string(index=False))

## Parte 5 — Guardar Resultados en Gold (Arquitectura Medallón)

Persistimos `gold_customer_segments` con el cluster asignado a cada cliente
para que los notebooks 06 (Business Insights) y equipos de BI puedan consumirla.

In [0]:
from pyspark.sql import functions as F

# Guardar resultados en tabla Delta Gold
df_segments = spark.createDataFrame(
    pdf[["id_cliente", "recency", "frequency", "monetary", "cluster"]]
)
df_segments = df_segments.withColumn("segment_timestamp", F.current_timestamp())

df_segments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bigdata.default.gold_customer_segments")

print("✅ Tabla gold_customer_segments guardada en Gold")
print(f"Total clientes: {df_segments.count():,}")
df_segments.groupBy("cluster").count().orderBy("cluster").show()